#  Détection d'Anomalies Stellaires par Autoencodeur Variationnel (VAE)

Ce projet utilise les données de Gaia DR3 pour entraîner un modèle capable de comprendre la distribution standard des étoiles. 

**Logique de détection :**
1. **Compression** : Le VAE apprend à compresser les caractéristiques d'une étoile (espace latent).
2. **Reconstruction** : Il tente de reconstruire l'étoile originale.
3. **Score d'Anomalie** : Si une étoile est trop différente de ce que le modèle a appris, l'erreur de reconstruction sera élevée, signalant un objet potentiellement exotique.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import logging

# Configuration du suivi
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

## 1. Préparation des données
On prépare les données astrométriques et photométriques. La normalisation est ici vitale pour la convergence du VAE.

In [2]:
def prepare_stellar_data(file_path):
    logging.info(f"📖 Lecture des archives : {file_path}")
    df = pd.read_parquet(file_path)
    
    # Caractéristiques clés pour définir une étoile 'standard'
    features = ['parallax', 'pmra', 'pmdec', 'phot_g_mean_mag', 'bp_rp']
    
    # Nettoyage : on élimine les mesures physiquement impossibles (ex: parallaxe négative)
    df_clean = df[(df['parallax'] > 0) & (df['phot_g_mean_mag'] > 0)].dropna(subset=features)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_clean[features])
    
    logging.info(f"✅ {len(X_scaled)} étoiles prêtes pour l'analyse.")
    return torch.tensor(X_scaled, dtype=torch.float32), scaler

X_tensor, data_scaler = prepare_stellar_data("data/gaia_rv_train_100k.parquet")

## 2. Architecture
Le VAE ne se contente pas de copier, il apprend une distribution statistique (Moyenne $\mu$ et Variance $\sigma$).

In [3]:
class StellarVAE(nn.Module):
    def __init__(self, input_dim=5, latent_dim=2):
        super().__init__()
        
        # Encodeur : réduction de dimension
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )
        
        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)
        
        # Décodeur : reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

model_vae = StellarVAE()
logging.info(" Architecture VAE initialisée.")

## 3. Apprentissage et Détection
Nous définissons la fonction de perte (Loss) qui équilibre la qualité de reconstruction et la régularité de l'espace latent.

In [4]:
def loss_function(recon_x, x, mu, logvar):
    mse = nn.functional.mse_loss(recon_x, x, reduction='sum')
    # Divergence KL : assure que l'espace latent est bien organisé
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return mse + kld

optimizer = torch.optim.Adam(model_vae.parameters(), lr=1e-3)
train_loader = DataLoader(TensorDataset(X_tensor), batch_size=128, shuffle=True)

logging.info("⏳ Début de l'entraînement...")
model_vae.train()
for epoch in range(20):
    total_loss = 0
    for batch in train_loader:
        x = batch[0]
        optimizer.zero_grad()
        recon, mu, logvar = model_vae(x)
        loss = loss_function(recon, x, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader.dataset):.4f}")

## 4. Analyse des Anomalies
Les étoiles ayant les erreurs de reconstruction les plus élevées sont marquées comme anomalies.

In [5]:
model_vae.eval()
with torch.no_grad():
    reconstructed, mu, logvar = model_vae(X_tensor)
    # Calcul du score d'anomalie (MSE individuelle)
    scores = torch.mean((reconstructed - X_tensor)**2, dim=1).numpy()

# On considère les 1% des étoiles les plus 'étranges' comme anomalies
threshold = np.percentile(scores, 99)
is_anomaly = scores > threshold

plt.figure(figsize=(10, 6))
plt.hist(scores, bins=50, color='royalblue', alpha=0.7)
plt.axvline(threshold, color='red', linestyle='--', label='Seuil Anomalie (Top 1%)')
plt.title("Distribution des scores de reconstruction (Surprise du modèle)")
plt.legend()
plt.show()

## 5. Cartographie des Objets Exotiques
Visualisons où se situent les anomalies par rapport à la population stellaire globale.

In [6]:
plt.figure(figsize=(12, 8))
plt.scatter(X_tensor[~is_anomaly, 3], X_tensor[~is_anomaly, 4], 
            s=1, c='gray', alpha=0.2, label='Étoiles Normales')
plt.scatter(X_tensor[is_anomaly, 3], X_tensor[is_anomaly, 4], 
            s=10, c='orange', label='Anomalies Détectées')

plt.xlabel("Magnitude G (Standardisée)")
plt.ylabel("Couleur BP-RP (Standardisée)")
plt.title("Localisation des Anomalies dans l'Espace Photométrique")
plt.legend()
plt.show()